# 문장과 화자 구분하기 (Speaker Diarization)

위스퍼는 음성을 잘 받아쓰지만 '누가' 말했는지는 구분하지 못한다.
pyannote 화자 분리 모델로 '누가 언제 말했는지'를 나누고, 결과를 표(CSV)로 정리한다.

**사전 준비 (한 번만):**
1. Hugging Face 로그인 후 아래 세 모델의 약관에 동의(Agree and access repository)
   - https://hf.co/pyannote/speaker-diarization-3.1
   - https://hf.co/pyannote/segmentation-3.0
   - https://hf.co/pyannote/speaker-diarization-community-1
2. `.env`에 `HUGGINGFACE_API_KEY=hf_...` (위 계정의 **Read** 토큰)
3. ffmpeg 설치: `winget install Gyan.FFmpeg`

In [ ]:
# pyannote.audio 설치. torch 계열은 버전이 어긋나면 'torchvision::nms' 오류가 나므로 맞춰서 고정한다.
%pip install pyannote.audio
%pip install torch==2.11.0 torchvision==0.26.0 torchaudio==2.11.0
# torchcodec은 Windows에서 ffmpeg DLL 문제로 오디오 디코딩을 깨뜨리므로 제거한다.
# (이 노트북은 아래 load_audio()로 직접 디코딩하므로 필요 없음)
%pip uninstall -y torchcodec

In [ ]:
import os
import glob
import shutil
from dotenv import load_dotenv

load_dotenv()
HUGGINGFACE_API_KEY = os.getenv("HUGGINGFACE_API_KEY")

# ffmpeg 자동 탐색: winget으로 설치한 ffmpeg 경로를 찾아 PATH에 추가한다.
# (책의 r"C:\ffmpeg\bin" 대신, 설치 위치를 자동으로 찾아 붙여 어느 터미널에서도 동작하게 함)
if shutil.which("ffmpeg") is None:
    _pkgs = os.path.join(os.environ.get("LOCALAPPDATA", ""),
                         "Microsoft", "WinGet", "Packages")
    _hits = glob.glob(os.path.join(_pkgs, "Gyan.FFmpeg*", "**", "ffmpeg.exe"),
                      recursive=True)
    if _hits:
        os.environ["PATH"] = os.path.dirname(_hits[0]) + os.pathsep + os.environ["PATH"]

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from pyannote.audio import Pipeline

pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1",
    token=HUGGINGFACE_API_KEY,
)

In [ ]:
import subprocess
import numpy as np
import torch


def load_audio(path, sr=16000):
    """ffmpeg로 오디오를 디코딩해 pyannote용 waveform 딕셔너리로 반환한다.
    최신 pyannote는 torchcodec으로 파일을 읽는데 Windows에서 DLL 문제로 실패할 수 있어,
    직접 디코딩한 파형을 넘겨 우회한다."""
    ffmpeg = shutil.which("ffmpeg")
    cmd = [ffmpeg, "-nostdin", "-i", path,
           "-f", "f32le", "-ac", "1", "-ar", str(sr), "-"]
    raw = subprocess.run(cmd, capture_output=True, check=True).stdout
    audio = np.frombuffer(raw, dtype=np.float32).copy()
    waveform = torch.from_numpy(audio).unsqueeze(0)  # (channel=1, time)
    return {"waveform": waveform, "sample_rate": sr}

In [ ]:
# 화자 분리 실행 후 RTTM 파일로 저장하기
audio_path = "./audio/싼기타_비싼기타.mp3"
output = pipeline(load_audio(audio_path))

# 최신 pyannote는 DiarizeOutput을 반환한다 → .speaker_diarization 이 기존 Annotation 객체
diarization = output.speaker_diarization

rttm_path = os.path.splitext(os.path.basename(audio_path))[0] + ".rttm"
with open(rttm_path, "w", encoding="utf-8") as rttm:
    diarization.write_rttm(rttm)

print("저장:", rttm_path)

In [ ]:
# RTTM을 CSV(데이터프레임)로 변환하기
import pandas as pd

df_rttm = pd.read_csv(
    rttm_path,          # rttm 파일 경로
    sep=" ",            # 구분자는 띄어쓰기
    header=None,        # 헤더는 없음
    names=["type", "file", "chnl", "start", "duration",
           "C1", "C2", "speaker_id", "C3", "C4"],
)
display(df_rttm)

In [ ]:
# 발언이 끝난 시간 추가하기 (start + duration = end)
df_rttm["end"] = df_rttm["start"] + df_rttm["duration"]
display(df_rttm)

In [ ]:
# 연속된 발화를 묶기 위해 number 열 추가하기
df_rttm["number"] = None       # number 열 만들고 None으로 초기화
df_rttm.at[0, "number"] = 0
display(df_rttm)

In [ ]:
# 화자 번호 매기기: 화자가 바뀌면 번호를 1 올리고, 같으면 유지한다.
for i in range(1, len(df_rttm)):
    if df_rttm.at[i, "speaker_id"] != df_rttm.at[i - 1, "speaker_id"]:
        df_rttm.at[i, "number"] = df_rttm.at[i - 1, "number"] + 1
    else:
        df_rttm.at[i, "number"] = df_rttm.at[i - 1, "number"]

display(df_rttm.head(10))

In [ ]:
# 같은 화자(연속 발화)끼리 묶어서 정리하기
df_rttm_grouped = df_rttm.groupby("number").agg(
    start=pd.NamedAgg(column="start", aggfunc="min"),
    end=pd.NamedAgg(column="end", aggfunc="max"),
    speaker_id=pd.NamedAgg(column="speaker_id", aggfunc="first"),
)
display(df_rttm_grouped)

In [ ]:
# 발화 시간(duration) 추가하고 인덱스 제거하기
df_rttm_grouped["duration"] = df_rttm_grouped["end"] - df_rttm_grouped["start"]
df_rttm_grouped = df_rttm_grouped.reset_index(drop=True)
display(df_rttm_grouped)

In [ ]:
# 화자 분리 결과를 CSV 파일로 저장하기
csv_path = os.path.splitext(os.path.basename(audio_path))[0] + "_rttm.csv"
df_rttm_grouped.to_csv(
    csv_path,
    sep=",",
    index=False,
)
print("저장:", csv_path)